Assignment 37: AstraDB RAG


In [1]:
!pip install -q langchain langchain-community langchain-astradb langchain-huggingface langchain-groq groq pypdf sentence-transformers cassio

## Imports

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_astradb import AstraDBVectorStore
from groq import Groq

from dotenv import load_dotenv
import os
load_dotenv()
ASTRA_DB_API_ENDPOINT = os.getenv("ASTRA_DB_API_ENDPOINT")
ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")



C:\Users\DS\AppData\Local\Temp\ipykernel_25104\871540788.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
#Test with 5 Questions
PDF_PATH = "DevAnand_Resume.pdf"
loader=PyPDFLoader(PDF_PATH)
documents=loader.load()
print("Pages:",len(documents))

#splitting now
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
split_docs=splitter.split_documents(documents)
print("Chunks:",len(split_docs))

Pages: 1
Chunks: 11


In [4]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model loaded")

vector_store = AstraDBVectorStore(collection_name="pdf_rag",embedding=embedding,api_endpoint=ASTRA_DB_API_ENDPOINT,token=ASTRA_DB_APPLICATION_TOKEN)
print("Connected to AstraDB")

print("Uploading documents...")

vector_store.add_documents(split_docs)
print("Documents stored successfully")

retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print("Retriever created")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\GenAI-Asssignements-TuteDude\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DS\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded
Connected to AstraDB
Uploading documents...
Documents stored successfully
Retriever created


In [7]:
#RAG cell
client=Groq(api_key=GROQ_API_KEY)
def ask_pdf(question):
    docs=retriever.invoke(question)
    context="\n\n".join([d.page_content for d in docs])
    prompt=f'''Answer ONLY from the context.
If the answer is not present, reply: I couldn't find this information in the uploaded PDF.
Context:{context}
Question:{question}'''

    res=client.chat.completions.create(model="llama-3.1-8b-instant",messages=[{"role":"user","content":prompt}],temperature=0)
    return res.choices[0].message.content, docs


In [8]:
#Test with 5 Questions  that is asked in the assignment
questions=[
"What technical skills are mentioned?",
"What projects are included?",
"What certifications are mentioned?",
"What programming languages are known?",
"What is the candidate's education?"
]

for q in questions:
    ans,docs=ask_pdf(q)
    print("="*60)
    print("Question:",q)
    print("Answer:",ans)
    print("Retrieved Pages:",[d.metadata.get("page") for d in docs])


Question: What technical skills are mentioned?
Answer: I couldn't find this information in the uploaded PDF.
Retrieved Pages: [0, 0, 0]
Question: What projects are included?
Answer: I couldn't find this information in the uploaded PDF.
Retrieved Pages: [0, 0, 0]
Question: What certifications are mentioned?
Answer: AWS Certified Cloud Practitioner 
AWS Certified Solutions Architect – Associate 
Salesforce Certified Agentforce Specialist 
Oracle Certified Foundations Associate 
GitHub Foundations 
NPTEL: Cloud Computing, Introduction to IoT
Retrieved Pages: [0, 0, 0]
Question: What programming languages are known?
Answer: Python, Go, Java
Retrieved Pages: [0, 0, 0]
Question: What is the candidate's education?
Answer: The candidate's education is a B.Tech in Information Technology from Karpagam Institute of Technology, from November 2022 to May 2026.
Retrieved Pages: [0, 0, 0]


In [9]:
#Out of Context questions to ask as it is mentioned in the assignment
q="Who won the IPL 2025?"
ans,_=ask_pdf(q)
print(ans)

I couldn't find this information in the uploaded PDF.


In [11]:
#Interactive question
while True:
    q=input("Ask a question (exit to stop): ")
    if q.lower()=="exit":
        break
    ans,_=ask_pdf(q)
    print(ans)


AWS Comprehend Medical, Bedrock, Personalize, Lambda, API Gateway, DynamoDB, S3, Amplify, CloudWatch.
I couldn't find this information in the uploaded PDF.
I couldn't find this information in the uploaded PDF.
Python
I couldn't find this information in the uploaded PDF.


## Observations

1. AstraDB stores embeddings in the cloud and is suitable for scalable RAG applications.
2. Session state helps preserve user interactions across requests.
3. FAISS is a local vector database, whereas AstraDB is a managed cloud vector database.
